In [1]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 634 (delta 152), reused 90 (delta 88), pack-reused 434 (from 3)
Receiving objects: 100% (634/634), 209.09 KiB | 19.01 MiB/s, done.
Resolving deltas: 100% (326/326), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 176 packages in 8.65s
Prepared 10 packages in 770ms
Uninstalled 4 packages in 214ms
Installed 10 packages in 45ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.1
 - holoviews==1.22.1
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - panel==1.9.0
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do n

In [9]:
import xgboost as xgb
import pandas as pd
import numpy as np
import cudf
import cuml
import seaborn as sns

from tensorflow import keras
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score
from cuml.ensemble import RandomForestClassifier as cuRF

In [3]:
df = pd.read_csv("dataset_completo_40036_v2.csv")

COLS_ELIMINAR = ['Tipo', 'L', 'B', 'A']
X = df.drop(COLS_ELIMINAR, axis=1)
y = df['Tipo']

class_counts = y.value_counts()
clases_validas = class_counts[class_counts >= 10].index
mask = y.isin(clases_validas)

X_filt = X[mask].copy()
y_filt = y[mask].copy()

clases_eliminadas = class_counts[class_counts < 10].index.tolist()

nan_cols = X_filt.columns[X_filt.isna().any()].tolist()


le = LabelEncoder()
y_enc = le.fit_transform(y_filt).astype(np.int32)


X_train, X_test, y_train, y_test = train_test_split(
    X_filt, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_imp).astype(np.float32)


In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(100,50,50),
    activation='tanh',
    solver='sgd',
    max_iter=1000,
    alpha = 0.001,
    random_state=42,
    learning_rate='adaptive',
)

from sklearn.utils.class_weight import compute_sample_weight
sw = compute_sample_weight('balanced', y_train)
mlp.fit(X_train_scaled, y_train)
predicciones = mlp.predict(X_test_scaled)

precision = accuracy_score(y_test, predicciones)
print(f"La precisión de la red es de: {precision * 100}%")

La precisión de la red es de: 99.92507492507492%


In [4]:
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cuda',
    random_state=42
)

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv = 5, scoring='accuracy')

grid_search.fit(X_train_scaled, y_train)

print(f"Mejor configuración: {grid_search.best_params_}")
print(f"Mejor score de validación: {grid_search.best_score_:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:06:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:06:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:06:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain

Mejor configuración: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Mejor score de validación: 0.9962


In [ ]:
import tensorflow as tf
import numpy as np

#Generar predicciones "suaves" del XGBoost sobre TODO el train
best_model = grid_search.best_estimator_
soft_labels = best_model.predict_proba(X_train_scaled)  # shape (n_samples, 7)

n_features = X_train_scaled.shape[1]
n_clases = soft_labels.shape[1]

#Definir red Keras
model_keras = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(n_clases, activation='softmax')
])

model_keras.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

#Entrenar imitando al XGBoost
model_keras.fit(
    X_train_scaled, soft_labels,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

#Evaluar vs el XGBoost original
y_pred_keras = np.argmax(model_keras.predict(X_test_scaled), axis=1)
y_pred_xgb   = best_model.predict(X_test_scaled)

print(f"Accuracy Keras:   {accuracy_score(y_test, y_pred_keras):.4f}")
print(f"Accuracy XGBoost: {accuracy_score(y_test, y_pred_xgb):.4f}")

#Guardar en formato Keras
model_keras.save('mejor_modelo.keras')

import json
with open('clases.json', 'w', encoding='utf-8') as f:
    json.dump(le.classes_.tolist(), f, ensure_ascii=False)
print(f"Clases guardadas ({len(le.classes_)}): {le.classes_.tolist()}")

Epoch 1/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.9475 - loss: 0.1708 - val_accuracy: 0.9788 - val_loss: 0.0553
Epoch 2/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9918 - loss: 0.0277 - val_accuracy: 0.9938 - val_loss: 0.0216
Epoch 3/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9950 - loss: 0.0205 - val_accuracy: 0.9950 - val_loss: 0.0172
Epoch 4/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9952 - loss: 0.0179 - val_accuracy: 0.9975 - val_loss: 0.0126
Epoch 5/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9967 - loss: 0.0145 - val_accuracy: 0.9984 - val_loss: 0.0107
Epoch 6/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9966 - loss: 0.0142 - val_accuracy: 0.9959 - val_loss: 0.0155
Epoch 7/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9974 - loss: 0.0139 - val_accuracy: 0.9991 - val_loss: 0.0092
Epoch 8/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9980 - loss: 0.0122 - val_accuracy: 0.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

#Constantes
ARCHIVO_ORIGEN = "Primeras muestras.csv"
ARCHIVO_SALIDA = "dataset_completo_40036_v3.csv"
COLUMNA_CLASE  = "Tipo"                           

# Los 18 canales espectrales
CANALES = ["A_410", "B_435", "C_460", "D_485", "E_510", "F_535",
           "G_560", "H_585", "R_610", "I_645", "S_680", "J_705",
           "T_730", "U_760", "V_810", "W_860", "K_900", "L_940"]

N_SINTETICAS   = 953        #Esto nos genera unas 40036 muestras
SIGMA          = 0.01       
RUIDO_RELATIVO = False      # Lo ponemos a false para que no mantengga la estructura estadística
SEMILLA        = 42

rng = np.random.default_rng(SEMILLA)

df = pd.read_csv(Path(ARCHIVO_ORIGEN), encoding="latin-1", sep=";", decimal=",")

X = df[CANALES].to_numpy(dtype=np.float64)       
y = df[COLUMNA_CLASE].to_numpy()


filas_sinteticas, etiquetas_sinteticas = [], []
for firma, etiqueta in zip(X, y):
    sigma_vec = SIGMA * np.abs(firma) if RUIDO_RELATIVO else SIGMA
    ruido = rng.normal(loc=0.0, scale=sigma_vec, size=(N_SINTETICAS, X.shape[1]))
    filas_sinteticas.append(firma + ruido)
    etiquetas_sinteticas.extend([etiqueta] * N_SINTETICAS)

X_sint = np.vstack(filas_sinteticas)

X_final = np.vstack([X, X_sint])
y_final = np.concatenate([y, etiquetas_sinteticas])

df_final = pd.DataFrame(X_final, columns=CANALES)
df_final[COLUMNA_CLASE] = y_final
df_final = df_final.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

df_final.to_csv(Path(ARCHIVO_SALIDA), index=False, encoding="utf-8-sig")

print(f"Firmas reales:       {len(df)}")
print(f"Muestras sintéticas: {len(X_sint)}")
print(f"Total dataset:       {len(df_final)}")
print(df_final[COLUMNA_CLASE].value_counts())

Firmas reales:       36
Muestras sintéticas: 34308
Total dataset:       34344
Tipo
Miel de milflores    13356
Miel Jaramago         5724
Miel de Bosque        5724
Miel de Retama        5724
Miel Sintética        3816
Name: count, dtype: int64
